In [34]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"daily": "rain_sum",
	"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "precipitation_probability", "weather_code", "wind_speed_10m", "cloud_cover", "surface_pressure", "dew_point_2m", "pressure_msl"],
	"timezone": "Europe/Berlin",
	"start_date": "2026-01-11",
	"end_date": "2026-04-03",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(3).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(6).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(7).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(8).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(9).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["weather_code"] = hourly_weather_code
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["pressure_msl"] = hourly_pressure_msl

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)
print(hourly_dataframe.info())

Coordinates: 52.52000045776367°N 13.419998168945312°E
Elevation: 38.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Hourly data
                           date  temperature_2m  relative_humidity_2m  \
0    2026-01-11 00:00:00+00:00             NaN                   NaN   
1    2026-01-11 01:00:00+00:00             NaN                   NaN   
2    2026-01-11 02:00:00+00:00             NaN                   NaN   
3    2026-01-11 03:00:00+00:00             NaN                   NaN   
4    2026-01-11 04:00:00+00:00             NaN                   NaN   
...                        ...             ...                   ...   
1987 2026-04-03 19:00:00+00:00        9.845500                  44.0   
1988 2026-04-03 20:00:00+00:00        9.695499                  48.0   
1989 2026-04-03 21:00:00+00:00        9.495500                  49.0   
1990 2026-04-03 22:00:00+00:00        8.895500                  57.0   
1991 2026-04-03 23:00:00+00:00        8.445499 

In [2]:
###
# Prepare Hopsworks project

import hopsworks
from dotenv import load_dotenv
import os

env = load_dotenv(".env")

project = hopsworks.login(
    project='mlops',  # Replace with your project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.environ["HOPSWORKS_API_KEY"]  # Get from Hopsworks UI > Account Settings > API Keys
)

fs = project.get_feature_store()


2026-04-04 20:37:45,625 INFO: Initializing external client
2026-04-04 20:37:45,625 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


/home/tobias/programmieren/aiops_mlops_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-04-04 20:37:46,795 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31926


In [60]:
# Feature Preparation and cleanup

feature_dataframe = hourly_dataframe.copy()

# Weather Variables could depend on the time of day
feature_dataframe["derived_hour"] = feature_dataframe["date"].apply(lambda x: x.hour)
# Weather Variables could depend on the month of the year
feature_dataframe["derived_month"] = feature_dataframe["date"].apply(lambda x: x.month)
# Generate Partion Key
feature_dataframe["partition"] = feature_dataframe["date"].apply(lambda x: f'{x.year}-{x.month}')
# Rain Probability could depend on the average humidity of the last 24 hours
feature_dataframe["timestamp"] = feature_dataframe["date"] # Copy to save column
feature_dataframe.set_index('date', inplace=True) # Needed for averaging over time later
feature_dataframe['relative_humidity_2m_24h_avg'] = feature_dataframe['relative_humidity_2m'].rolling('24h').mean()
# Rain Probability could depend on if the barometric pressure is rising or falling
feature_dataframe['pressure_msl_3h_pct_change'] = feature_dataframe['pressure_msl'].pct_change(periods=3)

# Find first valid index (during preparation pressure_msl_3h_pct_change returned the latest valid row)
first_non_nan_index = feature_dataframe['pressure_msl_3h_pct_change'].first_valid_index()

# Slice the DataFrame to keep only rows from the first non-NaN index onward
feature_dataframe = feature_dataframe.loc[first_non_nan_index:]


print(feature_dataframe.info())
print(feature_dataframe.head())



<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1544 entries, 2026-01-29 16:00:00+00:00 to 2026-04-03 23:00:00+00:00
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   temperature_2m                1544 non-null   float32            
 1   relative_humidity_2m          1544 non-null   float32            
 2   precipitation                 1544 non-null   float32            
 3   precipitation_probability     1544 non-null   float32            
 4   weather_code                  1544 non-null   float32            
 5   wind_speed_10m                1544 non-null   float32            
 6   cloud_cover                   1544 non-null   float32            
 7   surface_pressure              1544 non-null   float32            
 8   dew_point_2m                  1544 non-null   float32            
 9   pressure_msl                  1544 non-null   float32      

In [ ]:
fg = fs.create_feature_group("weather_prediction", version=1, primary_key=["timestamp"], event_time="timestamp", partition_key=["partition"], online_enabled=True)
fg.save(feature_dataframe)

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/31926/fs/20610/fg/37818


Uploading Dataframe: 100.00% |██████████| Rows 1544/1544 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: weather_prediction_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/31926/jobs/named/weather_prediction_1_offline_fg_materialization/executions


(Job('weather_prediction_1_offline_fg_materialization', 'PYSPARK'), None)

In [ ]:
# Concatenate with previous DataFrame (if available)

feature_group_name = "weather_prediction"

try:
    # Get all versions of the feature group
    fgs = fs.get_feature_groups()
    fg_versions = [fg for fg in fgs if fg.name == feature_group_name]

    if not fg_versions:
        print(f"Feature group '{feature_group_name}' does not exist.")
    else:
        # Get the latest version
        latest_version = max(fg_versions, key=lambda x: x.version).version
        fg = fs.get_feature_group(feature_group_name, version=latest_version)

        # Read the data into a pandas DataFrame
        old_feature_dataframe = fg.read()
        print(f"Data loaded successfully for version {latest_version}.")
except Exception as e:
    print(f"An error occurred: {e}")





Feature group 'weather_prediction' does not exist.


In [50]:
feature_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1544 entries, 2026-01-29 16:00:00+00:00 to 2026-04-03 23:00:00+00:00
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   temperature_2m                1544 non-null   float32
 1   relative_humidity_2m          1544 non-null   float32
 2   precipitation                 1544 non-null   float32
 3   precipitation_probability     1544 non-null   float32
 4   weather_code                  1544 non-null   float32
 5   wind_speed_10m                1544 non-null   float32
 6   cloud_cover                   1544 non-null   float32
 7   surface_pressure              1544 non-null   float32
 8   dew_point_2m                  1544 non-null   float32
 9   pressure_msl                  1544 non-null   float32
 10  derived_hour                  1544 non-null   int64  
 11  derived_month                 1544 non-null   int64  
 12  relative_humid